In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Procore — Work Order Contract Line Items
# MAGIC Pulls all Work Order Contracts for a project, then pulls all Line Items
# MAGIC (including Cost Code) for each contract, and prints the results.

# COMMAND ----------

import requests
import json
import time

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Auth — run the existing procore_auth notebook to get a token

# COMMAND ----------

auth_result = mssparkutils.notebook.run("procore_auth", 90)
auth_data = json.loads(auth_result)

ACCESS_TOKEN = auth_data["token"]
COMPANY_ID = auth_data["company_id"]

BASE_URL = "https://api.procore.com"

HEADERS = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Procore-Company-Id": str(COMPANY_ID),
}

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. (Optional) Look up available projects
# MAGIC Run this if you don't already know the Project ID you want. It lists
# MAGIC every project in the company so you can find the right `id`.

# COMMAND ----------

projects_url = f"{BASE_URL}/rest/v1.0/projects"
projects_resp = requests.get(
    projects_url,
    headers=HEADERS,
    params={"company_id": COMPANY_ID, "serializer_view": "compact", "per_page": 200},
)
projects_resp.raise_for_status()
for p in projects_resp.json():
    print(f"  id={p.get('id')}  name={p.get('name')}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Build the list of project IDs to loop over
# MAGIC Reuses the projects pulled in Step 2. By default this loops over
# MAGIC EVERY project in the company. Narrow it down if you only want
# MAGIC active projects, or a specific subset.

# COMMAND ----------

all_projects = projects_resp.json()

# To loop over everything:
project_ids = [p["id"] for p in all_projects]

# To narrow it down instead, comment the line above and use one of these:
# project_ids = [123456, 234567]  # specific IDs
# project_ids = [p["id"] for p in all_projects if "Active" in p.get("name", "")]

print(f"Will loop over {len(project_ids)} project(s)")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Helper: GET with pagination
# MAGIC Procore paginates with `page` / `per_page` query params. This helper
# MAGIC keeps requesting pages until a page comes back with fewer than
# MAGIC `per_page` results (i.e. the last page).

# COMMAND ----------

def get_all_pages(url, params=None, per_page=100, max_retries=5):
    params = dict(params or {})
    params["per_page"] = per_page

    all_results = []
    page = 1
    while True:
        params["page"] = page

        retries = 0
        while True:
            resp = requests.get(url, headers=HEADERS, params=params)

            if resp.status_code == 429:
                retries += 1
                if retries > max_retries:
                    resp.raise_for_status()
                wait_seconds = int(resp.headers.get("Retry-After", 30))
                print(f"  -> 429 rate limited, waiting {wait_seconds}s (retry {retries}/{max_retries})...")
                time.sleep(wait_seconds)
                continue

            if not resp.ok:
                # Surface Procore's actual error body instead of just the status code
                print(f"  -> {resp.status_code} on {resp.url}")
                print(f"  -> body: {resp.text[:500]}")
            resp.raise_for_status()
            break

        batch = resp.json()

        if not isinstance(batch, list):
            # Some Procore endpoints return a dict wrapper instead of a bare list
            raise ValueError(f"Expected a list response, got: {type(batch)} -> {batch}")

        all_results.extend(batch)

        if len(batch) < per_page:
            break
        page += 1

    return all_results

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Helper: pull contracts + line items for ONE project
# MAGIC A 404 here usually just means that project has no Work Order
# MAGIC Contracts module enabled / no contracts — we skip it rather than
# MAGIC blowing up the whole loop.

# COMMAND ----------

def get_line_items_for_project(project_id):
    contracts_url = f"{BASE_URL}/rest/v1.0/work_order_contracts"
    try:
        contracts = get_all_pages(contracts_url, params={"project_id": project_id})
    except requests.HTTPError as e:
        print(f"  [project {project_id}] skipped — {e}")
        return []

    project_line_items = []
    for contract in contracts:
        contract_id = contract["id"]
        line_items_url = f"{BASE_URL}/rest/v1.0/work_order_contracts/{contract_id}/line_items"
        try:
            line_items = get_all_pages(line_items_url, params={"project_id": project_id})
        except requests.HTTPError as e:
            print(f"  [project {project_id}, contract {contract_id}] skipped — {e}")
            continue

        for li in line_items:
            li["_project_id"] = project_id
            li["_work_order_contract_id"] = contract_id
            li["_work_order_contract_title"] = contract.get("title")

        project_line_items.extend(line_items)

    return project_line_items

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Loop over every project

# COMMAND ----------

all_line_items = []

for project_id in project_ids:
    print(f"Project {project_id}...")
    line_items = get_line_items_for_project(project_id)
    print(f"  -> {len(line_items)} line item(s)")
    all_line_items.extend(line_items)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Write raw results to the Bronze lakehouse
# MAGIC Append-only landing of the raw JSON payload per line item, plus
# MAGIC pull metadata (when it was pulled, which project/contract it came
# MAGIC from). No flattening/typing here on purpose — that belongs in the
# MAGIC silver/cleaning step. Adjust BRONZE_TABLE / BRONZE_PATH to match
# MAGIC your lakehouse.

# COMMAND ----------

from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp
import datetime

BRONZE_TABLE = "procore_work_order_line_items"  # resolves against the attached default lakehouse

pull_ts = datetime.datetime.utcnow().isoformat()

bronze_rows = [
    Row(
        project_id=li.get("_project_id"),
        work_order_contract_id=li.get("_work_order_contract_id"),
        line_item_id=li.get("id"),
        raw_json=json.dumps(li),  # full untouched payload, including nested cost_code
        pulled_at=pull_ts,
    )
    for li in all_line_items
]

if bronze_rows:
    bronze_df = spark.createDataFrame(bronze_rows)

    # Write as managed table (append-only):
    bronze_df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)

    # OR, if you write to a path instead of a table name, use this instead:
    # bronze_df.write.format("delta").mode("append").save(BRONZE_PATH)

    print(f"Wrote {bronze_df.count()} row(s) to {BRONZE_TABLE}")
else:
    print("No line items pulled this run — nothing written to bronze.")

StatementMeta(, 8d7a6986-6cfb-4e62-9fef-de6fecac5bbb, 3, Finished, Available, Finished, False)

  id=562949955001573  name=25-016 - 1100 Fulton Street
  id=562949954833574  name=24-011  - 11 ESSEX ST
  id=562949955118102  name=25-018 - 337A & 337B West Broadway Rehabilitaion Work
  id=562949955225798  name=25-021 - 360 Lexington 8th & 20th Floor
  id=562949955257421  name=25-020 - 549 Munroe Av
  id=562949955064640  name=25-017 - 64 MET OVAL PSC + 1410 MET STOREROOM
  id=562949955318524  name=26-023 - Boys & Girls Club
  id=562949955286476  name=26-022 - EMBANKMENT PHASE II
  id=562949955375634  name=26-026 - Embankment Phase III
  id=562949954973730  name=25-014 - Embankment + Revetment Apartments 270 & 310 10th Street NJ
  id=562949954973684  name=25-013 - Lillipvt 45 Renwick St
  id=562949954971827  name=25-012 - PCNA 711 11TH AVE
  id=562949953807489  name=1234 - Sandbox Test Project
  id=562949955365267  name=26-025 - SaunaLounge 45 South 3 Street, Brooklyn, NY
  id=562949953807474  name=Standard Project Template
  id=562949954507004  name=23-006 - SYMRISE - 15th & 16th Flr
